In [24]:
import pandas as pd
import re
import spacy
import nltk
import os
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

print("imported libraries successfully") 

imported libraries successfully


In [4]:
df = pd.read_csv("../data/processed/youtoxic_clean.csv")

print(f"Shape of the dataset: {df.shape}")
print(df.head())

Shape of the dataset: (997, 2)
                                                Text  IsToxic
0  If only people would just take a step back and...        0
1  Law enforcement is not trained to shoot to app...        1
2  \nDont you reckon them 'black lives matter' ba...        1
3  There are a very large number of people who do...        0
4  The Arab dude is absolutely right, he should h...        0


In [6]:
nlp = spacy.load("en_core_web_sm")
STOP = set(stopwords.words("english"))

print(f"Stopwords loaded: {len(STOP)}")
print(f"Spacy model : {nlp.meta['name']}")

Stopwords loaded: 198
Spacy model : core_web_sm


In [ ]:
def clean_text(text):
    """
    Clean basic text before lemmatizacion.
    Order important: first remove special characters, 
    then lemmatize and finally to lowercase.
    """
    text = str(text)                                    
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)     #Remove URLs
    text = re.sub(r"@\w+", " ", text)                      #Remove mentions
    text = re.sub(r"<[^>]+>", " ", text)                   #Remove HTML tags
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)             #looooool -> lool
    text = re.sub(r"[^a-zA-Z\s]", " ", text)               #Remove special characters and numbers
    text = re.sub(r"\s+", " ", text)                       #Remove extra whitespace
    text = text.lower().strip()                            

    return text

In [13]:
def lemmatize(text):
    """
    Lemmatize text using spacy.
    Remove stopwords and punctuation.
    """
    doc = nlp(text)
    tokens = [
        token.lemma_
        for token in doc
        if token.lemma_ not in STOP
        and len(token.lemma_) > 2
        and not token.is_space
    ]

    return " ".join(tokens)

In [14]:
def preprocess_text(text):
    """
    Pipeline complete : clean text and then lemmatize it.
    """
    text = clean_text(text)
    text = lemmatize(text)

    return text

In [15]:
examples = [
    "This is an example of a toxic comment! Visit http://example.com for more info.",
    "@user1 You are so stupid!!! <b>HTML tags</b> should be removed.",
    "Loooooool, this is hilarious!!! 😂😂😂",
    "Check out my website: www.example.com. It's amazing!",
    "I can't believe you said that! You're such an idiot."
]

print("Preprocessed examples:")
print("-" * 50)
for text in examples:
    result = preprocess_text(text)
    print(f"Original: {text}")
    print(f"Preprocessed: {result}")
    print("-" * 50)

Preprocessed examples:
--------------------------------------------------
Original: This is an example of a toxic comment! Visit http://example.com for more info.
Preprocessed: example toxic comment visit info
--------------------------------------------------
Original: @user1 You are so stupid!!! <b>HTML tags</b> should be removed.
Preprocessed: stupid html tag remove
--------------------------------------------------
Original: Loooooool, this is hilarious!!! 😂😂😂
Preprocessed: lool hilarious
--------------------------------------------------
Original: Check out my website: www.example.com. It's amazing!
Preprocessed: check website amazing
--------------------------------------------------
Original: I can't believe you said that! You're such an idiot.
Preprocessed: believe say idiot
--------------------------------------------------


In [19]:
print("Applying preprocessing to the dataset...")
df["text_clean"] = df["Text"].apply(preprocess_text)

empty = df[df["text_clean"].str.strip() == ""]
print(f"Rows with empty after preprocessing: {len(empty)}")
if len(empty) > 0:
    print(empty[["Text", "text_clean", "IsToxic"]])


Applying preprocessing to the dataset...
Rows with empty after preprocessing: 0


In [23]:
df = df[df["text_clean"].str.strip() != ""].reset_index(drop=True)

print(f"rows finally: {len(df)}")
print()

print("Comparative Original vs Processed: ")
print("-" * 50)
for _, row in df.sample(5, random_state=42).iterrows():
    print(f'Original:  {row["Text"][:80]}')
    print(f'Processed: {row["text_clean"]}')
    print(f'label:     {row["IsToxic"]}')
    print("-" * 50)

rows finally: 997

Comparative Original vs Processed: 
--------------------------------------------------
Original:  The music references sex, drugs, stealing, murder, etc.

...ya, pretty much all 
Processed: music reference sex drug steal murder etc pretty much rap music like
label:     0
--------------------------------------------------
Original:  That community should be marching for peace hand in hand for the girl that was k
Processed: community march peace hand hand girl kill sit home homework deserve least society let maybe sharpton lead
label:     0
--------------------------------------------------
Original:  Fucking savages! Protest sumthing real... Like how your government is robbing yo
Processed: fucking savage protest sumthe real like government rob sendin kid elegal war laugh stock america congrat
label:     1
--------------------------------------------------
Original:  death ti the white devil
Processed: death white devil
label:     1
-----------------------------------

In [25]:
os.makedirs("../data/preprocessed/", exist_ok=True)

df[['text_clean', 'IsToxic']].to_csv("../data/preprocessed/youtoxic_preprocessed.csv", index=False)

print(f"Saved: data/preprocessed/youtoxic_preprocessed.csv")
print(f"Rows saved: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(f"Sample saved:")
print(df[['text_clean', 'IsToxic']].head(3))

Saved: data/preprocessed/youtoxic_preprocessed.csv
Rows saved: 997
Columns: ['Text', 'IsToxic', 'text_clean']
Sample saved:
                                          text_clean  IsToxic
0  people would take step back make case anyone e...        0
1  law enforcement train shoot apprehend train sh...        1
2  reckon black life matter banner hold white cun...        1
